# mongodb

In [36]:
import os
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

from pathlib import Path
import json
from rich.tree import Tree
from rich import print as rprint

import os

In [37]:
client = MongoClient(
    host     = os.getenv("MONGO_HOST", "127.0.0.1"),
    port     = int(os.getenv("MONGO_PORT", 27017)),
    username = os.getenv("MONGO_USERNAME"),
    password = os.getenv("MONGO_PASSWORD"),
)

db         = client["job_market"]
collection = db["offres"]

print(client.list_database_names())

['admin', 'config', 'local']


In [38]:
def recuperer_recent(chemin_dossier)  -> list :
    '''
    Récupérer le chemin du document json le plus récent.
    Entrée : chemin du dossier visé
    Sortie : chemin du fichier horodaté le plus récent
    '''
    fichiers = []
    if chemin_dossier.exists():
        fichiers_ft = sorted(chemin_dossier.glob("*.json"))
        if fichiers_ft:
            fichiers.append(str(fichiers_ft[-1]))  # le plus récent  
    return fichiers[0]
#====================================================================================s
def charger_offres(chemin_fichier):
    '''
    Charger les offres du chemin du fichier json renseigné en paramètre
    '''
    try:
        with open(chemin_fichier, "r", encoding="utf-8") as f:
            offres = json.load(f)
        print(f"  {chemin_fichier} → {len(offres)} offres")
        return offres
    except Exception as e:
        print(f"  Erreur chargement {chemin_fichier} : {e}")

In [39]:
from pymongo import MongoClient
from pymongo.errors import BulkWriteError
import logging

def inserer_offres(
    offres:      list,
    collection,
    ordered:     bool = False
) -> dict:
    """
    Insère une liste d'offres dans MongoDB de façon sécurisée.

    - Crée un index unique sur "id" avant l'insertion pour eviter les doublons.
    - ordered=False : continue l'insertion même si une offre échoue.
    - ordered=True  : s'arrête à la première erreur.

    Retourne un rapport avec le nombre d'insertions réussies et les erreurs.
    """
    if not offres:
        logging.warning("Liste vide — rien à insérer.")
        return {"inseres": 0, "erreurs": 0}

    rapport = {"inseres": 0, "erreurs": 0, "detail_erreurs": []}

    # Créer l'index unique sur "id" avant toute insertion.
    # Si l'index existe déjà, MongoDB l'ignore silencieusement —
    # cette ligne est donc safe à appeler à chaque fois.
    collection.create_index("id", unique=True)

    try:
        resultat = collection.insert_many(offres, ordered=ordered)
        rapport["inseres"] = len(resultat.inserted_ids)
        logging.info(f"{rapport['inseres']} offres inserees avec succes.")

    except BulkWriteError as e:
        # Certaines insertions ont réussi, d'autres ont échoué
        rapport["inseres"] = e.details.get("nInserted", 0)
        rapport["erreurs"] = len(e.details.get("writeErrors", []))
        rapport["detail_erreurs"] = e.details.get("writeErrors", [])
        logging.warning(
            f"Insertion partielle : {rapport['inseres']} OK, "
            f"{rapport['erreurs']} erreurs."
        )

    except Exception as e:
        logging.error(f"Erreur inattendue lors de l'insertion : {e}")
        rapport["erreurs"] = len(offres)

    return rapport

In [40]:
# ── Étape 1 : Récupération du nom du fichier le plus récent ──────────────────────────────
chemin_fichier_raw_ft = recuperer_recent(Path("../data/raw/francetravail"))
chemin_fichier_raw_wttj = recuperer_recent(Path("../data/raw/welcometothejungle"))

chemin_fichier_processed_ft = recuperer_recent(Path("../data/processed/francetravail"))
chemin_fichier_processed_wttj = recuperer_recent(Path("../data/processed/welcometothejungle"))

chemin_fichier_normalise = recuperer_recent(Path("../data/processed/normalise"))

# ── Étape 2 : Récupération des offres dans Python (liste de dictionnaires) ──────────────
# Offres brutes issues de l'API France Travail et du scraping du site Welcome To The Jungle
offres_brutes_ft =  charger_offres(chemin_fichier_raw_ft)
offres_brutes_wttj =  charger_offres(chemin_fichier_raw_wttj)

# Offres parsees - traitement minimal (suppression des balises html, etc... 
offres_parsees_ft =  charger_offres(chemin_fichier_processed_ft)
offres_parsees_wttj =  charger_offres(chemin_fichier_processed_wttj)

# Offres normalisées
offres_normalisees =  charger_offres(chemin_fichier_normalise)


# Insérer
rapport = inserer_offres(offres_parsees_ft, collection, ordered=False)
print(rapport)
# → {"inseres": 95, "erreurs": 5, "detail_erreurs": [...]}

  ../data/raw/francetravail/data_engineer_20260407_142250.json → 578 offres
  ../data/raw/welcometothejungle/offres_20260407_142255.json → 1000 offres
  ../data/processed/francetravail/offres_20260407_142250.json → 578 offres
  ../data/processed/welcometothejungle/offres_20260407_142255.json → 1000 offres
  ../data/processed/normalise/offres_20260407_142335.json → 1578 offres
{'inseres': 578, 'erreurs': 0, 'detail_erreurs': []}
